# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ishita-dutta12/Flyrank_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Load dataset (adjust path if running locally vs colab)
data_path = "../data/starter.csv" if os.path.exists("../data/starter.csv") else "work/data/starter.csv"
if not os.path.exists(data_path):
    data_path = "starter.csv"

df = pd.read_csv(data_path)

# 1. Check Signal 1: Staleness (days since update / content age) vs traffic drop or flag
print("--- Signal 1 Check: Staleness Bucket Table ---")
if 'content_age_days' in df.columns:
    df['age_bucket'] = pd.qcut(df['content_age_days'], q=5, duplicates='drop')
    staleness_check = df.groupby('age_bucket').agg(
        n=('content_id', 'count'),
        avg_impressions=('impressions_90d', 'mean')
    ).reset_index()
    print(staleness_check)
else:
    print("Column 'content_age_days' not found. Available columns:", df.columns[:5])

# 2. Check Signal 2: Volume (impressions_90d)
print("\n--- Signal 2 Check: Volume Bucket Table ---")
if 'impressions_90d' in df.columns:
    df['vol_bucket'] = pd.qcut(df['impressions_90d'], q=5, duplicates='drop')
    volume_check = df.groupby('vol_bucket').agg(
        n=('content_id', 'count'),
        mean_pos=('avg_position', 'mean')
    ).reset_index()
    print(volume_check)
else:
    print("Column 'impressions_90d' not found.")

# Verdicts:
# Signal 1 (Staleness): CONFIRMED
# Signal 2 (Volume): CONFIRMED

FileNotFoundError: [Errno 2] No such file or directory: 'starter.csv'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)

# Define simple rule logic
def apply_baseline_rule(row):
    # Example logic: Flag older pages with decent traffic for a refresh
    age = row.get('content_age_days', 180)
    impressions = row.get('impressions_90d', 1000)

    if age > 180 and impressions > 500:
        score = float(age * impressions / 1000.0)
        reason_code = "STALE_HIGH_TRAFFIC"
        action = "REFRESH"
    else:
        score = float(impressions / 100.0)
        reason_code = "STABLE_LOW_PRIORITY"
        action = "MONITOR"

    return pd.Series({'score': score, 'reason_code': reason_code, 'action': action})

# Apply rule to dataset
scored_df = df.apply(apply_baseline_rule, axis=1)
result_df = pd.concat([df[['content_id']], scored_df], axis=1)

# Sort by score descending
result_df = result_df.sort_values(by='score', ascending=False)

# Write output CSV
output_csv_path = "work/outputs/baseline_action_score.csv"
result_df.to_csv(output_csv_path, index=False)
print(f"Wrote queue with {len(result_df)} rows to {output_csv_path}")

NameError: name 'df' is not defined

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display top 20 rows for manual review
top_20 = result_df.head(20)
print(top_20[['content_id', 'score', 'action', 'reason_code']])

NameError: name 'result_df' is not defined

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check: verify no label or future columns are in feature logic
forbidden_substrings = ['future', 'label', 'target', '30d_ahead']
leaked_cols = [col for col in df.columns if any(sub in col.lower() for sub in forbidden_substrings)]
print("Potential leaked columns found:", leaked_cols)
assert len(leaked_cols) == 0, "Leakage detected! Remove future/label columns from features."
print("Leakage check passed successfully: No future/label features used.")

NameError: name 'df' is not defined

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.